# VacinaBR-PNI: exemplos de consulta com DuckDB

Este notebook mostra como consumir os Parquets curados e o arquivo DuckDB gerados pela pipeline.

In [ ]:
!pip install -q duckdb pandas pyarrow

from pathlib import Path
import duckdb
import pandas as pd

ROOT = Path('/content/drive/MyDrive/VacinaBR-PNI')  # ajuste para Path('.') se estiver local
PARQUET_PATTERN = str(ROOT / 'data' / 'processed' / 'year=*' / 'month=*' / '*.parquet')
DB_PATH = ROOT / 'data' / 'vacinabr_pni.duckdb'

## Consulta direta nos Parquets

DuckDB lê os arquivos Parquet sem carregar tudo em memória.

In [ ]:
duckdb.sql(f"""
    SELECT
        uf_paciente,
        count(*) AS doses_aplicadas
    FROM read_parquet('{PARQUET_PATTERN}', union_by_name = true)
    GROUP BY uf_paciente
    ORDER BY doses_aplicadas DESC
    LIMIT 10
""").df()

## Vacinas por nome e sigla

In [ ]:
duckdb.sql(f"""
    SELECT
        sg_vacina,
        descricao_vacina,
        count(*) AS doses_aplicadas
    FROM read_parquet('{PARQUET_PATTERN}', union_by_name = true)
    GROUP BY sg_vacina, descricao_vacina
    ORDER BY doses_aplicadas DESC
    LIMIT 20
""").df()

## Qualidade dos registros por UF

In [ ]:
duckdb.sql(f"""
    SELECT
        uf_paciente,
        count(*) AS total_registros,
        round(100.0 * sum(CASE WHEN registro_completo THEN 1 ELSE 0 END) / count(*), 2) AS pct_completude,
        round(100.0 * sum(CASE WHEN idade_valida THEN 1 ELSE 0 END) / count(*), 2) AS pct_idade_valida
    FROM read_parquet('{PARQUET_PATTERN}', union_by_name = true)
    GROUP BY uf_paciente
    ORDER BY total_registros DESC
""").df()

## Usando o banco DuckDB pronto

Se `vacinabr_pni.duckdb` foi gerado, as tabelas de resumo já estão materializadas.

In [ ]:
con = duckdb.connect(str(DB_PATH))
con.sql('SELECT * FROM dataset_metadata').df()

In [ ]:
con.sql('SELECT * FROM municipality_vaccination_summary LIMIT 20').df()